In [ ]:
import pandas as pd
# df = pd.read_sas("../data/raw/P_OHXDEN.XPT")
# ctc_cols = [c for c in df.columns if c.endswith('CTC')]
# has_untreated_coronal = (
#     df[ctc_cols]
#     .apply(lambda row: (row == b'U').any(), axis=1)
# )
# has_root_caries = df["OHXRCAR"] == 1

# df["has_caries"] = (has_root_caries | has_untreated_coronal).astype(int)
# df.head()
df = pd.read_parquet("../data/processed/caries_features.parquet")
df.head()

In [ ]:
demo_df = pd.read_sas("../data/raw/P_DEMO.XPT")
demo_df.head()

In [ ]:
demo_df = demo_df[["SEQN", "RIDAGEYR", "RIAGENDR"]]

In [ ]:
demo_df['is_female'] = (demo_df['RIAGENDR']==2).astype(int)
demo_df = demo_df.drop('RIAGENDR', axis=1)

In [ ]:
df = df.merge(demo_df, on='SEQN', how='inner')

In [ ]:
df.shape
df[["RIDAGEYR", "is_female"]].isna().sum()

In [ ]:
X = df[
    [
        "RIDAGEYR",
        "is_female",
        "n_missing_teeth",
        "n_filled_teeth"
    ]
]

y = (df["has_root_caries"] == 1).astype(int)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, random_state=101)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000,solver="lbfgs")
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

import numpy as np
from sklearn.metrics import recall_score, precision_score, confusion_matrix, classification_report

threshold = 0.10
y_pred_tuned = (y_prob >= threshold).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))
